# 10 — Context engineering: why flat loops die

**What you'll learn**

- Measure a run's context growth with `shoplab.context.ContextLedger` and `count_tokens`, and see why a flat loop's token bill grows with the square of its steps
- Offload a plan or a bulky tool result with `offload` / `load_offload`, keeping a few-token handle in the window while the content lives on disk
- Use `run_subagent` as a context firewall so the parent sees a sub-agent's answer, never its whole transcript
- Fold the middle of a long conversation into one summary line with `context.compact`, and read the lossy tradeoff off the token count
- Pick the right one of four techniques — plan, offload, sub-agent, compact — for a given kind of overflow

*Time: ~3 min on a first live run; under a minute cached. Cost: ~$0.01. Cached reruns are free.*

In [ ]:
# === config (identical in every notebook) ===
import os, getpass
import litellm
from dotenv import load_dotenv              # pip install -e ".[obs]" if this fails

load_dotenv(".env")   # reads OPENROUTER_API_KEY / MODEL / STRONG_MODEL (see .env.example)

if not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("OpenRouter API key: ")

MODEL = os.environ.get("MODEL", "openrouter/deepseek/deepseek-v3.2")
STRONG_MODEL = os.environ.get("STRONG_MODEL", "openrouter/deepseek/deepseek-v4-flash")

# Per-notebook override: uncomment to ignore .env here (any LiteLLM provider works).
# MODEL = "openrouter/google/gemini-2.5-flash-lite"
# MODEL = "openai/gpt-4o-mini"              # direct OpenAI, uses OPENAI_API_KEY instead

TEMPERATURE = 0                             # the whole course runs at temperature 0
litellm.drop_params = True                  # ignore params a provider does not support
litellm.cache = litellm.Cache(type="disk", disk_cache_dir=".litellm_cache")  # reruns are ~free

### Phoenix observability (optional)

The triage run below goes through `shoplab.llm.complete`, so Phoenix traces its every step — a useful second view of the window filling turn by turn, the same growth this chapter measures offline. The offload, sub-agent, and compaction cells are pure token arithmetic and make no traced calls. Optional as ever: skip it and nothing else changes.

In [ ]:
# optional: Phoenix tracing (see notebook 03)
import obs

obs.enable_phoenix()

## The flat loop bills you twice

Every agent in this course so far has run the same loop: call the model, append its message, run the tools, append their results, repeat. The message list only ever grows — nothing is removed. That is fine for a five-step triage and fatal for a long one, because of an arithmetic the loop hides. The model is stateless: each step re-sends the *entire* history so far. Step one pays for one turn, step two pays for two, step _k_ pays for _k_. Over _N_ steps the tokens billed are 1 + 2 + ... + N — quadratic in the number of steps, though the final context is only linear in size.

So a run that ends at two thousand tokens did not cost two thousand tokens; it cost the area under the whole staircase. Left unmanaged, that same curve eventually walks into the context-window limit and the run simply stops. The fix is not a bigger window — it is spending the window deliberately. Anthropic calls this *context engineering*: "curating and maintaining the optimal set of tokens" an agent holds during inference ([Effective context engineering for AI agents](https://www.anthropic.com/engineering/effective-context-engineering-for-ai-agents)). This chapter builds `shoplab.context` — four tools for exactly that curation — and measures what each one saves.

## Watch one run's window fill

Before managing the window, measure it. `count_tokens` is the honest tape measure — `litellm.token_counter` under the model's own tokenizer, offline and free — and `ContextLedger` records a row each time you `observe` a message list, so a run's growth becomes a table you can read.

Run one ordinary triage from chapter 08 (a vip's opened, in-window return), then replay its final message list turn by turn to watch the window fill. The loop's `on_step` hook fires each step but is handed only the newest message; to see the whole growing list we walk `flat.messages` after the run and `observe` the prefix at each model turn.

In [ ]:
import json
from shoplab.context import count_tokens, ContextLedger
from shoplab.loop import run_agent
from shoplab.tools import standard_tools
from shoplab.world import load_tickets

SYSTEM = ("You are the operations desk agent for Larkspur Outfitters. Look up the order, "
          "the customer, and the one relevant policy, then decide. Call finish with "
          "decision, policy_id, and refund_usd. Decisions follow shop policy, not sympathy.")

def render_ticket(t):
    return (f"Ticket {t['ticket_id']} from {t['customer_id']} about order {t['order_id']}, "
            f"sku {t['sku']}, qty {t['qty']}, condition {t['item_condition']}, days since "
            f"delivery {t['days_since_delivery']}, requested {t['requested_action']}. "
            f"Customer writes: {t['reason_text']}")

ticket = next(t for s in load_tickets().values() for t in s if t["ticket_id"] == "TKT-2228")
flat = run_agent(render_ticket(ticket), standard_tools(), system=SYSTEM, max_steps=8)
print("decision:", flat.answer, "in", flat.steps, "steps,", len(flat.messages), "messages")

In [ ]:
ledger = ContextLedger()
prefix = []
for m in flat.messages:
    prefix.append(m)
    if m.get("role") == "assistant":          # a model turn: the list it was handed grew to here
        ledger.observe(list(prefix), label="model turn")

billed = 0
for row in ledger.growth():
    billed += row["tokens"]                    # each turn re-sends the whole prefix
    bar = "#" * (row["tokens"] // 40)
    print(f"turn {row['step'] + 1:>2}  {row['messages']:>2} msgs  {row['tokens']:>5} tok  {bar}")
print(f"\nfinal window: {ledger.peak()} tokens   |   billed across the run: {billed} tokens")

> **What you should see:** the window climbs every turn — a couple hundred tokens at the first lookup, several times that by the finish — because each turn carries all the earlier ones. The final window is under two thousand tokens, yet the run was *billed* for the sum of every staircase step, several times the peak. (The real API bill is larger still: the tool schemas ride along on every call too, which `count_tokens` over the message list alone does not include.) Push the step count up and that gap widens as the square — the flat loop's core problem, now a number instead of a worry.

## Offload the plan, keep only a handle

The first technique is the simplest: text that does not need to be *read* every turn does not need to sit in the window every turn. A plan is the clean example. The agent writes its plan once, then works the steps — it needs the plan on disk, not re-tokenized on every call. `offload` writes any blob to a file under `artifacts/offload/` and hands back a small handle — `{"ref", "chars", "preview"}` — and that handle is all that stays in context. `load_offload` fetches the full text back on the one turn that actually needs it.

In [ ]:
from shoplab.context import offload, load_offload

plan = ("Triage playbook for TKT-2228 (opened-item refund):\n"
        "1. get_order ORD-7313 - confirm it is delivered, read the line item and unit\n"
        "   price, and note the shipping charged in case damaged-in-transit applies.\n"
        "2. get_customer CUST-01 - read the loyalty tier; a vip has the 10% restocking\n"
        "   fee waived (pol-loyalty), everyone else pays it on opened goods.\n"
        "3. search_policy 'opened item restocking refund' - expect pol-restocking; confirm\n"
        "   the fee is 10% and that store credit would waive it if the customer preferred.\n"
        "4. check the return window: days_since_delivery must be <= 30 (pol-returns), or the\n"
        "   refund path is denied regardless of the item's condition.\n"
        "5. compute with calc: full item value for a vip, else round(value * 0.90, 2).\n"
        "6. finish with decision, policy_id, and refund_usd - and nothing else.")

handle = offload(plan, tag="plan")
print("handle kept in context:", handle)
print("plan inline:", count_tokens(plan), "tokens   handle:", count_tokens(json.dumps(handle)), "tokens")
print("\nloaded back on demand:", load_offload(handle)[:60], "...")

> **What you should see:** the handle is three fields — a file path, a character count, and a 120-character preview — so it costs on the order of seventy tokens whether the plan is six lines or six hundred; the plan inline already runs several times heavier and only grows from there. `load_offload(handle)` returns the exact text back, byte for byte. The plan persisted *outside* the window; the window only ever held a pointer to it.

## A big tool result belongs on disk

The same move rescues the window from a tool that returns too much. Search a knowledge base, dump a table, read a long document — the result can be larger than everything else in the conversation combined, and the agent usually needs one fact from it, not the whole thing. Offload the bulk, keep the handle, and let the agent `load_offload` only if it must.

This is the idea behind MemGPT, which treats the context window like the RAM of an operating system: a small, precious tier backed by cheap external storage, with information *paged* between them on demand.

*MemGPT: Towards LLMs as Operating Systems* (Packer et al., 2023, [arXiv:2310.08560](https://arxiv.org/abs/2310.08560)).

| Paper concept | Plain English | Where it lives in code |
|---|---|---|
| Context window as main memory | the tokens the model can see right now — scarce and expensive | the live `messages` list |
| External storage tier | cheap space holding what will not fit | files under `artifacts/offload/` |
| Paging between tiers | move data in and out on demand, not all at once | `offload` writes out; `load_offload` reads back |
| Memory-pressure signal | act while the window is filling, not after it overflows | the `ContextLedger` growth curve |

In [ ]:
from shoplab.world import load_policies

dump = json.dumps(load_policies())             # all 12 policy docs, the whole knowledge base
inline = count_tokens([{"role": "tool", "content": dump}])

ref = offload(dump, tag="policies")
kept = count_tokens([{"role": "tool", "content": json.dumps(ref)}])
print(f"policy dump: {ref['chars']} chars, {inline} tokens inline")
print(f"as a handle: {kept} tokens kept  ->  {inline - kept} saved ({100 * (inline - kept) // inline}%)")
print("round-trips clean:", load_offload(ref) == dump)

> **What you should see:** the twelve policy documents run to well over a thousand tokens inline; kept as a handle they cost under a hundred — a saving north of 90%. The full text is unchanged on disk (`load_offload` reproduces it exactly), it just no longer rides along on every model call. That is the difference between a knowledge base the agent *can reach* and one it is *forced to carry*.

## Sub-agents as context firewalls

Offloading moves *data* out of the window; a sub-agent moves *work* out of it. When a step needs its own multi-turn investigation — research a policy corner, reconcile a mismatched order — running it inline pours every lookup and every tool result into the parent's context, which the parent then carries forever after. A sub-agent runs that investigation in a fresh, separate message list and returns only its conclusion. The parent never sees the intermediate turns: the sub-agent's transcript is a firewall its window sits behind.

`run_subagent` is exactly that — it runs a full `run_agent` loop and hands back only `{answer, steps, stop_reason}`. Run the same research task both ways, inline and firewalled, and compare what the parent would have to hold.

In [ ]:
from shoplab.context import run_subagent

def research_tools():                          # read-only: no finish tool, so it answers in prose
    t = standard_tools()
    return {n: t[n] for n in ("get_order", "get_customer", "search_policy", "calc")}

SUB = ("You are a returns-policy researcher. Use the lookup tools to gather what you need, "
       "then answer in one or two sentences. You have no finish tool; just write your answer.")
task = ("For ticket TKT-2205 (order ORD-7312, customer CUST-07, sku LK-1016, opened, returned "
        "18 days after delivery, refund requested): look up the order and the customer, find "
        "the restocking policy, and state the refund the customer is owed.")

inline = run_agent(task, research_tools(), system=SUB, max_steps=8)
fire = run_subagent(task, research_tools(), system=SUB, max_steps=8)
print(f"inline transcript: {len(inline.messages)} messages, {count_tokens(inline.messages)} tokens")
print(f"firewalled result: {count_tokens([{'role': 'tool', 'content': json.dumps(fire)}])} tokens")
print("parent receives ->", fire)

> **What you should see:** the inline research runs to a dozen-plus messages and well over a thousand tokens — every `get_order`, `get_customer`, and `search_policy` result, in full. The firewalled result the parent receives is around a hundred tokens: the answer, the step count, and the stop reason, nothing else. Same work, same conclusion; the parent's window is spared the entire investigation. (The two runs need not take the same number of steps — an agent at temperature 0 is still not deterministic — but the transcript always dwarfs the result.)

## Compaction: fold the middle, keep the ends

Plans, offloads, and sub-agents all keep bulk *out* of the window. Sometimes the bulk is already *in* — a long run whose own history has outgrown what you want to carry. Compaction is the answer of last resort: summarize the middle of the conversation and drop the turns it replaces. `compact` keeps the leading system messages and the last few turns verbatim — recency is where the live task lives — and folds everything between into one summary line. It never orphans a tool result from the assistant call that produced it: if the cut would sever that pair, the kept window extends to keep them together.

Here is the canonical body, pasted from `src/shoplab/context.py` between its sentinels — byte-identical to the package, the chapter's build-then-import contract. The default summarizer is deliberately LLM-free (roles plus a snippet), so compaction is testable offline; pass your own `summarizer` for a model-written précis.

In [ ]:
def _default_summary(messages) -> str:         # the default, LLM-free summarizer
    return " | ".join(f"{m.get('role', '?')}: {str(m.get('content') or '')[:80]}"
                      for m in messages)

# >>> shoplab.context.compact
def compact(messages, *, keep_last=4, summarizer=None) -> list:
    """Fold the middle of a conversation into one summary line.

    Keep the leading system message(s) at the head and the last ``keep_last``
    messages verbatim; replace everything between with a single
    ``{"role": "system", "content": "[compacted N messages] " + summary}``.
    ``summarizer(list)->str`` defaults to a deterministic role+snippet join (no
    LLM, so this is testable offline). Never orphan a trailing tool result from
    its assistant tool_call: if the cut would sever one, the kept window extends
    to keep them together. Returns a NEW list."""
    summarizer = summarizer or _default_summary
    head = 0
    while head < len(messages) and messages[head].get("role") == "system":
        head += 1
    system_head, body = messages[:head], messages[head:]
    cut = max(0, len(body) - keep_last)
    while 0 < cut < len(body) and body[cut].get("role") == "tool":
        cut -= 1  # would orphan a tool result from its assistant tool_call
    middle, tail = body[:cut], body[cut:]
    if not middle:
        return list(messages)
    note = {"role": "system",
            "content": f"[compacted {len(middle)} messages] " + summarizer(middle)}
    return system_head + [note] + tail
# <<< shoplab.context.compact

`compact` is now `shoplab.context.compact`. Give it the flat run's own message list — the triage from the top of the chapter — keep the last four turns verbatim, and measure the window before and after.

In [ ]:
before = count_tokens(flat.messages)
compacted = compact(flat.messages, keep_last=4)
after = count_tokens(compacted)
print(f"before: {len(flat.messages)} messages, {before} tokens")
print(f"after:  {len(compacted)} messages, {after} tokens   ({100 * (before - after) // before}% smaller)")
print("\nthe summary line that replaced the middle:")
print(compacted[1]["content"][:150], "...")

> **What you should see:** the conversation collapses to a handful of messages — the system head, one summary line, and the last four turns — and the token count drops by roughly a third to two-thirds. The summary line is prefixed `[compacted N messages]` and carries a snippet of each folded turn. This is the one lossy technique of the four: detail in the middle is gone for good, which is why `keep_last` protects the recent turns the agent is still reasoning over. Compact when you must; offload before you must.

## Four techniques, four situations

The four are not interchangeable — each answers a different question about *why* the window is full. Reach for the one that matches the pressure.

| Technique | Use it when | What leaves the window | Lossy? |
|---|---|---|---|
| Plan artifact | reasoning or a checklist is re-read rarely but held constantly | the plan text, replaced by a handle | no |
| Offload | a tool returns more than the agent needs to keep | the bulky result, replaced by a handle | no |
| Sub-agent | a step needs its own multi-turn investigation | the whole sub-transcript, replaced by its answer | no |
| Compact | the live conversation itself has already grown too long | the middle turns, replaced by a summary | yes |

The first three keep bulk out of the window and lose nothing — prefer them. Compaction is the fallback for when the conversation you must keep is itself too big, and it is the only one that forgets.

## Recap

| Concept | One-liner |
|---|---|
| Flat loop is quadratic | a stateless model re-sends the whole history each step; N steps bill 1+2+...+N tokens. |
| `count_tokens` | offline tape measure (`litellm.token_counter`), the honest number behind the growth curve. |
| `ContextLedger` | `observe` a message list per step; `growth` and `peak` turn a run into a readable table. |
| `offload` / `load_offload` | write a blob under `artifacts/offload/`, keep a few-token handle, fetch it back on demand. |
| Plan artifact | text re-read rarely but held constantly belongs on disk, not in every call. |
| Filesystem offload | a bulky tool result paged out to storage — MemGPT's window-as-RAM idea in miniature. |
| `run_subagent` | a context firewall: the parent sees `{answer, steps, stop_reason}`, never the sub-transcript. |
| `context.compact` | fold the middle into one summary, keep the system head and last turns; the only lossy technique. |
| Four techniques | plan, offload, sub-agent, compact — pick by why the window is full; the first three lose nothing. |

## Exercises

1. Make the quadratic bite. Push `max_steps` up and point the flat agent at a ticket that forces more lookups (or a task that fans out several tool calls per step), replaying the run through a `ContextLedger`. Tabulate `billed` against step count: does the total track the square of the steps while the final window grows only linearly?
2. Write a decisions-only summarizer. The default summarizer keeps a snippet of every folded turn; write one that keeps only the *decisions* — the tool calls made and their results — and drops the model's prose reasoning. Pass it to `compact` on the flat run and compare both the token count and the readability of the summary line against the default.
3. Triage from a handle. Offload the full customer table with `offload`, hand the agent only the handle plus a `load_customer(id)` tool that reads one row back via `load_offload`, and run a triage. Confirm the decision matches the inline-table run while the window never holds more than the one customer the ticket is about.

**Next up:** chapter 11 stops hand-rolling these techniques and builds a *deep agent* on top of them — plan, filesystem, and sub-agents wired together as one harness (LangGraph and deepagents) that manages its own context over a task too long for any single window.